# 08 — 中文场景扩展

**多语言清洗的特殊挑战**

英文的 heuristic 规则不能直接用于中文，原因：

1. **分词问题**：中文没有空格分词，"word count" 规则失效
   - 英文: "The quick brown fox" = 4 词
   - 中文: "快速的棕色狐狸" = 7个字，但算"1词"（因为无空格）
   
2. **标点系统不同**：
   - 英文句末标点：`. ? !`
   - 中文句末标点：`。 ？ ！ …`
   
3. **停用词不同**：英文停用词（the, a, is）完全不适用于中文

4. **简繁体问题**：同一内容可能以简体或繁体存在，需要统一化处理

参考资料：
- FineWeb-2（多语言版）：https://github.com/huggingface/fineweb-2
- FineWeb-zhtw（繁体中文版）：https://github.com/mtkresearch/fineweb-zhtw

In [1]:
import sys, re
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

# 中文文本处理工具
def chinese_word_count(text):
    """中文文本的"词"数估算（基于字符数，而非空格分词）"""
    # 方法1: 直接字符数（最简单）
    return len(text)

def chinese_sentence_tokenize(text):
    """中文句子分割"""
    # 中文句末标点
    return re.split(r'[。！？…]+', text)

def chinese_stop_ratio(text):
    """中文常用字占比（代替英文停用词检查）"""
    # 高频汉字（代替停用词）
    common_chars = set('的了在是我有和人这中大为上个国我以要他时来用们生到作地于出就分对成会可主发年动同工也能下过子说产种面而方后多定行学法所民得经十三之进着等部度家电力里如水化高自二理起小物现实加量都两体制机当使点从业本去把性好应开它合还因由其些然前外天政四日那社义事平形相全表间样与关各重新线内数正心力理而产品政如'.split())
    count = sum(1 for c in text if c in common_chars)
    return count / max(len(text), 1)

# 示例中文文本
chinese_samples = [
    "人工智能技术正在深刻改变我们的生活方式。机器学习算法通过大量数据的训练，能够识别图像、理解语音、翻译文字。",
    "买买买！限时特惠！全场五折！快来抢购！数量有限！先到先得！点击链接！",
    "研究表明，规律运动对身体健康有显著益处。每周至少150分钟的中等强度有氧运动，可降低心血管疾病风险约35%。",
    "http://www.click.here/buy?utm_source=spam&product=1234 立即购买 优惠码 折扣",
]

print("中文文本分析示例:")
for i, text in enumerate(chinese_samples):
    print(f"\n样本 {i+1}: {text[:50]}...")
    print(f"  字符数: {len(text)}")
    print(f"  句子数: {len([s for s in chinese_sentence_tokenize(text) if s.strip()])}")

中文文本分析示例:

样本 1: 人工智能技术正在深刻改变我们的生活方式。机器学习算法通过大量数据的训练，能够识别图像、理解语音、翻译...
  字符数: 53
  句子数: 2

样本 2: 买买买！限时特惠！全场五折！快来抢购！数量有限！先到先得！点击链接！...
  字符数: 34
  句子数: 7

样本 3: 研究表明，规律运动对身体健康有显著益处。每周至少150分钟的中等强度有氧运动，可降低心血管疾病风险约...
  字符数: 54
  句子数: 2

样本 4: http://www.click.here/buy?utm_source=spam&product=...
  字符数: 66
  句子数: 1


## 中文 Heuristic 规则适配

需要将英文规则改写为中文适配版本：

In [2]:
class ChineseQualityFilter:
    """
    中文文本质量过滤器（英文规则的中文适配版）
    """
    
    # 中文句末标点
    CHINESE_TERMINAL_PUNCT = set('。！？…')
    
    def __init__(
        self,
        min_chars: int = 100,          # 最少字符数（替代英文词数规则）
        max_chars: int = 200000,
        min_terminal_punct_ratio: float = 0.3,  # 句末标点结尾行的最低比例
        max_punctuation_ratio: float = 0.2,    # 标点占总字符比例上限
    ):
        self.min_chars = min_chars
        self.max_chars = max_chars
        self.min_terminal_ratio = min_terminal_punct_ratio
        self.max_punct_ratio = max_punctuation_ratio
    
    def check(self, text):
        """返回 (passes, fail_reason)"""
        
        # 规则 1: 字符数范围（中文用字符数而非词数）
        n_chars = len(text.strip())
        if n_chars < self.min_chars:
            return False, f'too_short:{n_chars}<{self.min_chars}'
        if n_chars > self.max_chars:
            return False, f'too_long:{n_chars}>{self.max_chars}'
        
        # 规则 2: 中文字符比例（至少包含一定比例的汉字）
        chinese_chars = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
        if chinese_chars / n_chars < 0.3:
            return False, f'low_chinese_ratio:{chinese_chars/n_chars:.2f}'
        
        # 规则 3: 标点比例（过高说明是符号堆砌）
        punct_chars = sum(1 for c in text if c in '，。！？；：""''（）【】、…')
        if punct_chars / n_chars > self.max_punct_ratio:
            return False, f'high_punct_ratio:{punct_chars/n_chars:.2f}'
        
        return True, ''


# 测试中文过滤器
cn_filter = ChineseQualityFilter()
print("中文质量过滤器测试:")
for i, text in enumerate(chinese_samples):
    passes, reason = cn_filter.check(text)
    status = '✅ 通过' if passes else f'❌ 过滤（{reason}）'
    print(f"  样本 {i+1}: {status}")

中文质量过滤器测试:
  样本 1: ❌ 过滤（too_short:53<100）
  样本 2: ❌ 过滤（too_short:34<100）
  样本 3: ❌ 过滤（too_short:54<100）
  样本 4: ❌ 过滤（too_short:66<100）


## 简繁体处理

中文数据中存在简体（Simplified）和繁体（Traditional）两种字符系统。
统一化处理是重要的预处理步骤，尤其对于去重（同一内容简繁体不同 → 哈希不同 → 无法去重）。

FineWeb-zhtw 专门为繁体中文开发了适配版本，保留繁体的语言习惯，
同时应用了重新校准的质量规则阈值。

In [3]:
# 简繁体转换（如果 opencc 可用）
try:
    import opencc
    converter_s2t = opencc.OpenCC('s2t')   # 简体 → 繁体
    converter_t2s = opencc.OpenCC('t2s')   # 繁体 → 简体
    
    simplified = "机器学习是人工智能的一个子领域"
    traditional = converter_s2t.convert(simplified)
    back_to_simplified = converter_t2s.convert(traditional)
    
    print("简繁体转换示例:")
    print(f"  简体: {simplified}")
    print(f"  繁体: {traditional}")
    print(f"  回转: {back_to_simplified}")
    print(f"  ✅ 简繁互转正确: {simplified == back_to_simplified}")
    
except ImportError:
    print("opencc 未安装（可选依赖）")
    print("安装方式: pip install opencc-python-reimplemented")
    print()
    print("简繁体统一化对去重的重要性:")
    text_simplified = "机器学习算法"
    text_traditional = "機器學習算法"  # 手动模拟繁体
    import xxhash
    hash_s = xxhash.xxh64(text_simplified.encode()).hexdigest()
    hash_t = xxhash.xxh64(text_traditional.encode()).hexdigest()
    print(f"  简体哈希: {hash_s}")
    print(f"  繁体哈希: {hash_t}")
    print(f"  相同内容不同哈希: {hash_s != hash_t} ← 这就是为什么需要统一化")

opencc 未安装（可选依赖）
安装方式: pip install opencc-python-reimplemented

简繁体统一化对去重的重要性:
  简体哈希: 4ee5179919950f17
  繁体哈希: d39e6348fe12af71
  相同内容不同哈希: True ← 这就是为什么需要统一化


In [4]:
# 中文 SEO / 营销号内容检测
def detect_chinese_spam(text):
    """检测中文 SEO 堆砌和营销号模板"""
    
    # 关键词密度（过高说明 SEO 堆砌）
    spam_keywords = ['购买', '优惠', '折扣', '限时', '特惠', '点击', '立即', 
                      '加微信', '私信', '免费', '赚钱', '日入万元']
    keyword_count = sum(text.count(kw) for kw in spam_keywords)
    keyword_density = keyword_count / max(len(text) / 10, 1)  # 每10字的关键词数
    
    # 重复标点（!!! ？？？）
    repeated_punct = len(re.findall(r'[！？!?]{2,}', text))
    
    # 网址/联系方式密度
    url_count = len(re.findall(r'http|www|微信|QQ|电话|手机', text))
    
    spam_score = keyword_density + repeated_punct * 0.1 + url_count * 0.2
    is_spam = spam_score > 1.0
    
    return {'is_spam': is_spam, 'spam_score': round(spam_score, 3), 
            'keyword_count': keyword_count, 'repeated_punct': repeated_punct}

print("中文 SEO/营销号检测:")
for i, text in enumerate(chinese_samples):
    result = detect_chinese_spam(text)
    status = '🚫 营销号' if result['is_spam'] else '✅ 正常'
    print(f"  样本 {i+1}: {status} | spam_score={result['spam_score']}")

中文 SEO/营销号检测:
  样本 1: ✅ 正常 | spam_score=0.0
  样本 2: ✅ 正常 | spam_score=0.882
  样本 3: ✅ 正常 | spam_score=0.0
  样本 4: 🚫 营销号 | spam_score=1.006


In [5]:
# 结论
print("=" * 60)
print("  中文场景扩展 — 总结")
print("=" * 60)
print()
print("关键适配要点（英文规则 → 中文版）:")
print("  词数规则 → 字符数规则（Chinese chars count）")
print("  英文停用词 → 高频汉字比例检查")
print("  英文句末标点 → 中文句末标点（。！？…）")
print("  PII 正则 → 中国电话格式（+86, 1xx-xxxx-xxxx）")
print()
print("额外中文特有处理:")
print("  简繁体统一化（去重前必须）")
print("  SEO/营销号关键词检测")
print("  中文 fastText langid（识别中文方言/日文干扰）")
print()
print("参考资源:")
print("  FineWeb-2: https://github.com/huggingface/fineweb-2")
print("  FineWeb-zhtw: https://github.com/mtkresearch/fineweb-zhtw")

  中文场景扩展 — 总结

关键适配要点（英文规则 → 中文版）:
  词数规则 → 字符数规则（Chinese chars count）
  英文停用词 → 高频汉字比例检查
  英文句末标点 → 中文句末标点（。！？…）
  PII 正则 → 中国电话格式（+86, 1xx-xxxx-xxxx）

额外中文特有处理:
  简繁体统一化（去重前必须）
  SEO/营销号关键词检测
  中文 fastText langid（识别中文方言/日文干扰）

参考资源:
  FineWeb-2: https://github.com/huggingface/fineweb-2
  FineWeb-zhtw: https://github.com/mtkresearch/fineweb-zhtw
